# Pipeline d'analyse de l'action

Notebook Jupyter pédagogique : téléchargement, nettoyage, analyse des moyennes mobiles et détection des signaux BUY/SELL.

### Installation des bibliothèques nécessaires
Cette cellule installe les bibliothèques Python requises pour l'analyse financière (yfinance), la manipulation de données (pandas) et la visualisation (matplotlib).

In [ ]:
!pip install yfinance pandas matplotlib

### Importation des modules
Cette cellule importe les modules Python `yfinance`, `pandas` et `matplotlib.pyplot` pour pouvoir les utiliser dans le reste du notebook.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

### Téléchargement des données historiques de l'action AAPL
Cette cellule télécharge les données historiques de l'action Apple (symbole boursier 'AAPL') pour une période spécifique (du 1er janvier 2024 au 1er janvier 2026) et ajuste automatiquement les prix (par exemple, pour les divisions d'actions). Ensuite, elle affiche les cinq premières lignes du DataFrame pour un aperçu.

In [ ]:
# Télécharger les données historiques de l'action
ticker_symbol = 'NVDA' # @param ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA'] {type:"string"}
data = yf.download(
    ticker_symbol,
    start="2024-01-01",
    end="2026-01-01",
    auto_adjust=True
)

# Afficher les 5 premières lignes
display(data.head())

### Aperçu des dimensions et des informations du DataFrame
Cette cellule affiche le nombre de lignes et de colonnes du DataFrame `data` (`data.shape`) et fournit un résumé des informations sur les colonnes, y compris les types de données et le nombre de valeurs non nulles (`data.info()`).

In [ ]:
# Nombre de lignes et de colonnes
print("Dimensions :", data.shape)

# Informations générales
data.info()

### Vérification des valeurs manquantes
Cette cellule vérifie la présence de valeurs manquantes (nulles) dans chaque colonne du DataFrame et affiche leur somme. Cela permet d'identifier les colonnes qui pourraient nécessiter un nettoyage des données.

In [ ]:
# Vérifier s'il existe des valeurs nulles
data.isnull().sum()

### Suppression des lignes avec des valeurs manquantes
Cette cellule supprime toutes les lignes du DataFrame qui contiennent au moins une valeur manquante. Ensuite, elle vérifie à nouveau s'il reste des valeurs nulles pour confirmer le nettoyage.

In [ ]:
# Supprimer les lignes contenant des valeurs manquantes
data = data.dropna()

# Vérifier à nouveau
data.isnull().sum()

### Statistiques descriptives des données
Cette cellule calcule et affiche les statistiques descriptives de base pour chaque colonne numérique du DataFrame (moyenne, écart-type, minimum, maximum, quartiles, etc.). L'utilisation de `.T` transpose le tableau pour une meilleure lisibilité.

In [ ]:
data.describe().T

### Calcul de la moyenne mobile sur 100 périodes (MA_100)
Cette cellule calcule la moyenne mobile sur 100 jours (MA_100) pour la colonne 'Close' (prix de clôture) et ajoute cette nouvelle série au DataFrame. Elle affiche ensuite les dernières valeurs du prix de clôture et de la MA_100.

In [ ]:
# Moyenne mobile sur 100 observations
data["MA_100"] = data["Close"].rolling(window=100).mean()

# Afficher les dernières valeurs
data[["Close", "MA_100"]].tail()

### Calcul de la moyenne mobile sur 200 périodes (MA_200)
Cette cellule calcule la moyenne mobile sur 200 jours (MA_200) pour la colonne 'Close' et l'ajoute également au DataFrame. Elle affiche les dernières valeurs du prix de clôture, de la MA_100 et de la MA_200.

In [ ]:
# Moyenne mobile sur 200 observations
data["MA_200"] = data["Close"].rolling(window=200).mean()

# Afficher les dernières valeurs
data[["Close", "MA_100", "MA_200"]].tail()

### Visualisation des prix et des moyennes mobiles
Cette cellule génère un graphique affichant l'évolution du prix de clôture de l'action AAPL, ainsi que ses moyennes mobiles sur 100 et 200 jours. Cela permet de visualiser les tendances et les croisements des moyennes mobiles.

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    data["Close"],
    label="AAPL"
)

plt.plot(
    data["MA_100"],
    label="MA_100 : moyenne des 100 observations"
)

plt.plot(
    data["MA_200"],
    label="MA_200 : moyenne des 200 observations"
)

plt.title("Moyennes mobiles - AAPL")
plt.xlabel("Date")
plt.ylabel("Prix")

plt.legend()
plt.grid()
plt.show()

In [ ]:
# Télécharger les données historiques de l'action
ticker_symbol = 'NVDA' # @param ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA'] {type:"string"}
data = yf.download(
    ticker_symbol,
    start="2024-01-01",
    end="2026-01-01",
    auto_adjust=True
)

# Afficher les 5 premières lignes
display(data.head())

### Génération des signaux d'achat/vente
Cette cellule crée une colonne 'Signal' dans le DataFrame. Un signal de `1` est attribué si la MA_100 est supérieure à la MA_200 (potentiel signal d'achat), et un ` -1` si la MA_100 est inférieure à la MA_200 (potentiel signal de vente). Elle affiche les 10 dernières lignes des colonnes pertinentes.

In [ ]:
# Créer une colonne Signal
data["Signal"] = 0

# Si MA_100 > MA_200 → signal 1
data.loc[data["MA_100"] > data["MA_200"], "Signal"] = 1

# Si MA_100 < MA_200 → signal -1
data.loc[data["MA_100"] < data["MA_200"], "Signal"] = -1

data[["Close", "MA_100", "MA_200", "Signal"]].tail(10)

### Détection des croisements de signaux
Cette cellule calcule la différence entre le signal actuel et le signal précédent ('Crossover'). Un changement de signal de -1 à 1 donnera une différence de 2 (signal d'achat), et un changement de 1 à -1 donnera une différence de -2 (signal de vente). Elle affiche les 20 dernières lignes des colonnes pertinentes.

In [ ]:
# Différence entre le signal actuel et le signal précédent
data["Crossover"] = data["Signal"].diff()

data[[
    "Close",
    "MA_100",
    "MA_200",
    "Signal",
    "Crossover"
]].tail(20)

### Identification et comptage des signaux BUY et SELL
Cette cellule filtre le DataFrame pour identifier les jours où un signal d'achat (`Crossover == 2`) ou un signal de vente (`Crossover == -2`) est détecté. Elle imprime ensuite le nombre total de signaux d'achat et de vente trouvés.

In [ ]:
# Détecter les signaux BUY
buy_signals = data[data["Crossover"] == 2]

# Détecter les signaux SELL
sell_signals = data[data["Crossover"] == -2]

print("Nombre de signaux BUY :", len(buy_signals))
print("Nombre de signaux SELL :", len(sell_signals))

### Affichage détaillé des signaux BUY et SELL
Cette cellule affiche les détails (prix de clôture, MA_100, MA_200) pour chaque date où un signal d'achat a été détecté, puis fait de même pour les signaux de vente.

In [ ]:
print("===== SIGNAUX BUY =====")
print(buy_signals[["Close", "MA_100", "MA_200"]])

print("\n===== SIGNAUX SELL =====")
print(sell_signals[["Close", "MA_100", "MA_200"]])

### Visualisation des signaux d'achat et de vente sur le graphique
Cette cellule génère un graphique complet qui superpose le prix de clôture, les moyennes mobiles (MA_100, MA_200) et marque explicitement les points d'achat (triangles verts) et de vente (triangles rouges) détectés.

In [ ]:
plt.figure(figsize=(14, 7))

# Prix AAPL
plt.plot(
    data["Close"],
    label=ticker_symbol
)

# Moyenne mobile 100
plt.plot(
    data["MA_100"],
    label="MA_100"
)

# Moyenne mobile 200
plt.plot(
    data["MA_200"],
    label="MA_200"
)

# Signaux BUY
plt.scatter(
    buy_signals.index,
    buy_signals["Close"],
    marker="^",
    s=100,
    label="BUY"
)

# Signaux SELL
plt.scatter(
    sell_signals.index,
    sell_signals["Close"],
    marker="v",
    s=100,
    label="SELL"
)

plt.title(f"{ticker_symbol} - Moyennes mobiles et signaux")
plt.xlabel("Date")
plt.ylabel("Prix")
plt.legend()
plt.grid()

plt.show()

### Affichage des dernières données avec tous les indicateurs
Cette cellule affiche les 20 dernières lignes du DataFrame `data`, y compris toutes les colonnes calculées (prix de clôture, MA_100, MA_200, Signal, Crossover), pour un aperçu final des données traitées.

In [ ]:
data.tail(20)

### Résumé du pipeline d'analyse
Cette cellule imprime un résumé textuel de l'ensemble du pipeline d'analyse de l'action AAPL, indiquant les différentes étapes qui ont été complétées avec succès.

In [ ]:
print("====================================")
print("        PIPELINE ",ticker_symbol)
print("====================================")

print("1. Données téléchargées       ✓")
print("2. Données vérifiées          ✓")
print("3. Données nettoyées          ✓")
print("4. MA_100 calculée            ✓")
print("5. MA_200 calculée            ✓")
print("6. Signaux générés            ✓")
print("7. Croisements détectés       ✓")
print("8. Graphique créé             ✓")
print("9. Statistiques calculées     ✓")